In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "iga_core").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
DATA_DIR = PROJECT_ROOT / "paper_tables"
FIGURES_DIR = PROJECT_ROOT / "paper_figures"

from iga_core import (
    make_knots,
    elements_spans,
    quadrature_grid,
    basis_ders_on_quad_grid,
    gauss_legendre,
    L2_projection,
    Mass_Matrix,
    Stiffness_Matrix,
    plot_field_1d,
    sol_plot,
    int_approx,
    NIA,
    plots_splines,
    Diff_coef,
    varying_diff_coeff,
    plot_heatmap,
    plot_surface,
)
import matplotlib.pyplot as plt
import numpy as np
from scipy.sparse import csc_matrix, csr_matrix, linalg as sla
from numpy import linspace, zeros, sin, pi, array, cos, exp, eye, dot
from scipy.sparse import csr_matrix, linalg as sla
from mpl_toolkits.mplot3d import Axes3D

In [ ]:
capacity = 1e-5
Lp       = 3.2e-7            # Electrode thickness [m] (320 nm)
sa       = 1e-4              # Electrode area [mÂ²] (1 cmÂ²)
a_max    = 2.33e4            # Max activity [mol/mÂ³]
c0       = 1
F        = 96485             # Faraday constant [C/mol]

In [ ]:
# --- Spatial domain setup ---
ne     = 8
grid   = linspace(0, 1, ne+1)
p      = 3 # degree  of spline
nders  = 1
knots            = make_knots(grid, p, False) # the knots vector
nbasis           = len(knots)-p-1 # number of basis functions
nelements        = len(grid)-1
spans            = elements_spans(knots, p) # last basis function non-vanishing

In [ ]:
U, W            = gauss_legendre(p)   # gauss legendre rule ------> returns  arrays of size (p+1)

points, weights = quadrature_grid(grid, U, W)
basis           = basis_ders_on_quad_grid( knots, p, points, nders, normalize=False ) # is a tensor of order 4 ( ne X (p+1) X nders X nq)

B       = zeros((nbasis, nbasis))
Stiffness_Matrix(nelements, p, spans, basis, weights, points, B)


A       = zeros((nbasis, nbasis))
Mass_Matrix(nelements, p, spans, basis, weights, points, A)

### *Initial solution*

$c(x,0) = cste$

In [ ]:
U_0x = lambda x: c0
U_0  = L2_projection(knots, p, U_0x)
plot_field_1d(knots, p, U_0 , 'r--')
plt.legend(['$c_{Li^+}(x,0)$'])
plt.show()

In [ ]:
D_ref = 1.76e-15  
data = 'DNN'

# --- Temporal domain setup ---
nb_min           = 1
T_0              = Lp**2/D_ref         # seconds 
T                = nb_min*60/T_0       # [no unit]
dt               = 5e-3             # [no unit]
nt               = int(T / dt)  # number of time points including t=0
dt_r             = dt * T_0

t_vals = np.linspace(0.0, T, nt)
t_phys = t_vals * T_0          # convert to seconds

In [ ]:
# 1. Define the temporal knot vector
ne_t   = 8
p_t    = 2
grid_t = np.linspace(0, T, ne_t + 1)
knots_t = make_knots(grid_t, p_t, periodic=False)
nbasis_t           = len(knots_t) - p_t - 1 # number of basis functions

# 2. Evaluate B-spline basis functions on the normalized time grid
t_norm, Phi_t = plots_splines(knots_t, p_t, nt)   # nt = number of PDE time steps
n_params = Phi_t.shape[1]

fig, ax = plt.subplots(figsize=(5.1, 3.6))
for i in range(Phi_t.shape[1]):
    ax.plot(t_norm, Phi_t[:, i], linewidth=1.2)
ax.scatter(grid_t, np.zeros_like(grid_t), color="black", s=14, zorder=3, label="Breakpoints")
ax.set_xlabel("Nondimensional time")
ax.set_ylabel(r"Temporal B-spline basis $N_i(t)$")
ax.legend(frameon=False, loc="upper right")
fig.tight_layout()
bspline_figure_path = FIGURES_DIR / "temporal_bspline_basis.pdf"
fig.savefig(bspline_figure_path, format="pdf", bbox_inches="tight")
print(f"Saved temporal B-spline figure: {bspline_figure_path}")
plt.show()
plt.close(fig)
print("n_params",n_params)
print("nbasis_t", nbasis_t)
print(t_norm.shape)


In [ ]:
e0    = np.zeros(nbasis)
e0[0] = 1.0
nx    = 101
y = np.linspace(0.0, 1.0, nx)

M = A + dt * B
lu = sla.splu(csr_matrix(M))

c_basis = np.zeros((n_params, nt, nx))
H_basis = np.zeros((n_params, nt))   # <-- add this

for k in range(n_params):
    cn = np.zeros(nbasis)              # zero IC for basis response
    phi_k = Phi_t[:, k]                # nondimensional time basis

    # nondimensional boundary flux
    j_k = (phi_k * capacity * Lp) / (F * sa * D_ref * a_max)

    for n in range(1, nt):
        rhs = np.dot(A, cn) - dt * j_k[n] * e0
        cn  = lu.solve(rhs)

        # coefficients -> nondimensional field
        Q, _ = sol_plot(knots, p, cn, nx=nx)
        c_basis[k, n, :] = Q[:, 0]     # already nondimensional

        # health functional H_k(t_n) = c_avg,k(t_n) - c_s,k(t_n)
        c_avg_k = int_approx(nelements, p, spans, basis, weights, cn)
        H_basis[k, n] = c_avg_k - c_basis[k, n, 0]



In [ ]:
def reconstruct_c_and_H(alpha):

    c = np.zeros((nt, nx))
    c_IC = c0 * np.ones(nx)
    avg_list = []
    T_charged = None
    SOC = [0]
    
    for n in range(nt):
        c[n, :] = c_IC
        for k in range(n_params):
            c[n, :] += alpha[k] * c_basis[k, n, :]
        c_avg = np.trapezoid(c[n, :], y)
        avg_list.append(c_avg)
        if c_avg <= 0.6:            
            T_charged = n * dt_r
            SOC.append(T_charged)
    
    H = np.trapezoid(c, y, axis=1) - c[:, 0]   # H(t) = c_avg(t) - c_s(t)
    return c, H, avg_list, SOC[0]

## Maximal feasible solution

In [ ]:
jmax  = 60
H_max = 0.1  # Same prescribed safety constraint as the refined policy

alpha1 = jmax * np.ones(n_params)
H1     = H_basis.T @ alpha1

idx_s = np.argmax(H1 >= H_max)
if H1[idx_s] < H_max:
    idx_s = None

t_s_nd = t_vals[idx_s]     # because t_vals spans [0, T]
t_s_phys = t_phys[idx_s]

print(t_s_phys)
print(t_s_nd)


In [ ]:
K_pre, K_mix, K_post = [], [], []

for k in range(n_params):
    tL = knots_t[k]
    tR = knots_t[k + p_t + 1]
    if tR <= t_s_nd:
        K_pre.append(k)
    elif tL >= t_s_nd:
        K_post.append(k)
    else:
        K_mix.append(k)

U = K_mix + K_post
print("K_pre",K_pre)
print("K_mix",K_mix)
print("K_post",K_post)



In [ ]:
alpha_opt = np.zeros(n_params)
alpha_opt[K_pre] = jmax

# --- indices for constraints ---
idx_B = np.arange(idx_s, nt)      # boundary arc collocation indices (temporarily full tail)
idx_1 = np.arange(0, idx_s+1)     # phase-1 collocation indices

# --- (A) boundary constraint: H(t)=H_max for t>=ts ---
A_H = H_basis[U][:, idx_B].T
b_H = H_max - (H_basis[K_pre][:, idx_B].T @ alpha_opt[K_pre])

# --- (B) phase-1 constraint: j_amp(t)=jmax for t<=ts ---
A_J = Phi_t[idx_1][:, U]
b_J = jmax*np.ones(len(idx_1)) - Phi_t[idx_1][:, K_pre] @ (jmax*np.ones(len(K_pre)))

# --- stack with a large weight to enforce phase-1 strongly ---
lam = 1e4
A_constraints = np.vstack([A_H, lam*A_J])
b = np.concatenate([b_H, lam*b_J])

alpha_U, *_ = np.linalg.lstsq(A_constraints, b, rcond=None)
alpha_opt[U] = alpha_U

j_amp = Phi_t @ alpha_opt
print("j_amp min/max:", j_amp.min(), j_amp.max())



In [ ]:

data_refined = np.load(DATA_DIR / "refined_linear_solution.npz", allow_pickle=True)
params = data_refined["params"].item()

expected_params = {
    "Lp": Lp,
    "c0": c0,
    "a_max": a_max,
    "D_ref": D_ref,
    "H_max": H_max,
    "T": T,
}
for name, expected in expected_params.items():
    actual = params[name]
    if not np.isclose(actual, expected):
        raise ValueError(
            f"Refined data mismatch for {name}: saved={actual}, current={expected}. "
            "Run health_constrained_refined_policy.ipynb first."
        )

j_ref = data_refined["j_amp"]
def reconstruct_H_from_current(j_profile):
    """Re-simulate a current profile and evaluate H = c_avg - c_s."""
    cn = U_0.copy()
    H_profile = np.zeros(len(j_profile))
    H_profile[0] = int_approx(nelements, p, spans, basis, weights, cn) - cn[0]
    lu_profile = sla.splu(csc_matrix(A + dt * B))

    for n in range(1, len(j_profile)):
        j_nd = (j_profile[n] * capacity * Lp) / (F * sa * D_ref * a_max)
        rhs = np.dot(A, cn) - dt * j_nd * e0
        cn = lu_profile.solve(rhs)
        H_profile[n] = int_approx(nelements, p, spans, basis, weights, cn) - cn[0]

    return H_profile

H_ref = reconstruct_H_from_current(j_ref)

In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

# ============================================================
# PaperPlaza / IEEE-safe Matplotlib settings
# Avoid Type 3 fonts in exported PDF figures
# ============================================================
mpl.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "text.usetex": False,          # important: avoid LaTeX-generated Type 3 fonts
    "font.family": "serif",
    "font.serif": ["DejaVu Serif"],
    "mathtext.fontset": "dejavuserif",
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 9,
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
    "figure.dpi": 300,
    "savefig.dpi": 300,
})

# ============================================================
# Reconstruct current (unrefined) solution
# ============================================================
c_opt, H_opt, avg_list, SOC = reconstruct_c_and_H(alpha_opt)

# ============================================================
# Load refined data
# ============================================================
data_refined = np.load(DATA_DIR / "refined_linear_solution.npz", allow_pickle=True)
params = data_refined["params"].item()

j_ref = data_refined["j_amp"]
H_ref = reconstruct_H_from_current(j_ref)

if "t_phys" in data_refined.files:
    t_phys_ref = data_refined["t_phys"]
else:
    t_phys_ref = t_phys

# ============================================================
# Figure 4: refined vs unrefined
# ============================================================
fig, axes = plt.subplots(2, 1, figsize=(3.5, 4.2), sharex=True)

# ------------------------------------------------------------
# (a) charging input
# ------------------------------------------------------------
ax = axes[0]

ax.plot(
    t_phys,
    j_amp,
    linewidth=1.8,
    label="Unrefined"
)

ax.plot(
    t_phys_ref,
    j_ref,
    "--",
    linewidth=1.8,
    label="Refined"
)

ax.set_ylabel(r"Charging input $j(t)$")
ax.text(
    0.02,
    0.95,
    "(a)",
    transform=ax.transAxes,
    fontsize=9,
    fontweight="bold",
    va="top"
)
ax.legend(frameon=False, loc="best")
ax.grid(True, alpha=0.3)

# ------------------------------------------------------------
# (b) health trajectory
# ------------------------------------------------------------
ax = axes[1]

ax.plot(
    t_phys,
    H_opt,
    linewidth=1.8,
    label="Unrefined"
)

ax.plot(
    t_phys_ref,
    H_ref,
    "--",
    linewidth=1.8,
    label="Refined"
)

ax.axhline(
    H_max,
    linestyle="--",
    color="black",
    linewidth=1.2,
    label=r"$H_{\max}$"
)

ax.set_xlabel("Time [s]")
ax.set_ylabel(r"$H(t)=c_{\mathrm{avg}}(t)-c_s(t)$")
ax.text(
    0.02,
    0.95,
    "(b)",
    transform=ax.transAxes,
    fontsize=9,
    fontweight="bold",
    va="top"
)

ax.legend(frameon=False, loc="best")
ax.grid(True, alpha=0.3)

# ============================================================
# Final layout and export
# ============================================================
fig.tight_layout()

# Save as PaperPlaza-safe PDF
fig.savefig(FIGURES_DIR / "figure_04_temporal_refinement.pdf", format="pdf")

plt.show()
plt.close(fig)